# Imports

In [0]:
from pyspark.sql.functions import trim, when, length, lit, col, row_number, lower as lower_spark, concat_ws, coalesce, current_timestamp, sha2, sum as sum_spark, lower as lower_spark, upper as upper_spark, countDistinct, first, dense_rank, isnan
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

BRONZE_LAPS_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.laps"
)

SILVER_LAPS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.laps"
)

SILVER_RACE_RESULTS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.race_results"
)

SILVER_DRIVERS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.drivers"
)

# Metodos

In [0]:
def clean_string(column_name: str):
    value = trim(col(column_name))

    return (
        when(length(value) == 0, lit(None).cast("string"))
         .otherwise(value)
    )

In [0]:
def ns_to_ms(column_name: str):
    return (
        when(
            col(column_name).isNotNull(),
            (col(column_name) / 1_000_000).cast("long")
        )
    )

def double_is_missing(column_name: str):
    return (
        col(column_name).isNull()
        |
        isnan(col(column_name))
    )


def double_to_int(column_name: str):
    return (
        when(
            double_is_missing(column_name),
            lit(None).cast("int")
        )
        .otherwise(
            col(column_name).cast("int")
        )
    )


def clean_double(column_name: str):
    return (
        when(
            double_is_missing(column_name),
            lit(None).cast("double")
        )
        .otherwise(
            col(column_name).cast("double")
        )
    )

In [0]:
def get_latest_laps_snapshot():
    bronze_df = spark.table(
        BRONZE_LAPS_TABLE
    )

    snapshot_window = (
        Window
        .partitionBy(
            "season",
            "round"
        )
        .orderBy(
            col("_source_file_modification_time")
                .desc_nulls_last(),

            col("_ingested_at")
                .desc_nulls_last()
        )
    )

    return (
        bronze_df
        .withColumn(
            "_snapshot_rank",
            dense_rank().over(
                snapshot_window
            )
        )
        .filter(
            col("_snapshot_rank") == 1
        )
        .drop("_snapshot_rank")
    )

In [0]:
def get_race_participants():
    race_results = (
        spark.table(
            SILVER_RACE_RESULTS_TABLE
        )
        .select(
            "season",
            "round",
            "driver_number",
            "driver_id",
            "constructor_id"
        )
    )

    drivers = (
        spark.table(
            SILVER_DRIVERS_TABLE
        )
        .select(
            "driver_id",
            "abbreviation"
        )
    )

    participants = (
        race_results
        .join(
            drivers,
            on="driver_id",
            how="left"
        )
    )

    # Un número no puede identificar a dos pilotos
    # dentro de la misma carrera.
    duplicate_numbers = (
        participants
        .groupBy(
            "season",
            "round",
            "driver_number"
        )
        .count()
        .filter(
            col("count") > 1
        )
        .count()
    )

    if duplicate_numbers > 0:
        raise ValueError(
            "Race participant mapping contains duplicate "
            "(season, round, driver_number) values."
        )

    return participants

In [0]:
def enrich_laps_with_participants(df):
    participants = get_race_participants()

    laps = (
        df
        .withColumn(
            "_driver_number_normalized",
            clean_string("DriverNumber")
        )
        .withColumn(
            "_driver_abbreviation_normalized",
            upper_spark(
                clean_string("Driver")
            )
        )
    )

    joined = (
        laps.alias("laps")
        .join(
            participants.alias("participant"),
            (
                col("laps.season")
                == col("participant.season")
            )
            &
            (
                col("laps.round")
                == col("participant.round")
            )
            &
            (
                col(
                    "laps._driver_number_normalized"
                )
                == col(
                    "participant.driver_number"
                )
            ),
            "left"
        )
    )

    # ------------------------------------------------------
    # Every lap must resolve to a race participant
    # ------------------------------------------------------

    unresolved_count = (
        joined
        .filter(
            col(
                "participant.driver_id"
            ).isNull()
        )
        .count()
    )

    if unresolved_count > 0:
        raise ValueError(
            f"{unresolved_count} laps could not be mapped "
            "to silver.race_results."
        )

    # ------------------------------------------------------
    # Cross-check Driver abbreviation
    # ------------------------------------------------------

    abbreviation_mismatch_count = (
        joined
        .filter(
            col(
                "laps._driver_abbreviation_normalized"
            )
            !=
            col(
                "participant.abbreviation"
            )
        )
        .count()
    )

    if abbreviation_mismatch_count > 0:
        raise ValueError(
            f"{abbreviation_mismatch_count} laps have a "
            "Driver abbreviation inconsistent with "
            "silver.drivers."
        )

    return (
        joined
        .select(
            "laps.*",

            col(
                "participant.driver_id"
            ).alias("driver_id"),

            col(
                "participant.constructor_id"
            ).alias("constructor_id")
        )
        .drop(
            "_driver_number_normalized",
            "_driver_abbreviation_normalized"
        )
    )

In [0]:
def transform_laps(df):
    return (
        df
        .select(
            # ----------------------------------------------
            # Race / participants
            # ----------------------------------------------

            col("season")
                .cast("int")
                .alias("season"),

            col("round")
                .cast("int")
                .alias("round"),

            col("driver_id"),

            clean_string("DriverNumber")
                .alias("driver_number"),

            col("constructor_id"),

            # ----------------------------------------------
            # Lap identity
            # ----------------------------------------------

            double_to_int("LapNumber")
                .alias("lap_number"),

            double_to_int("Stint")
                .alias("stint_number"),

            # ----------------------------------------------
            # Timing
            # ----------------------------------------------

            ns_to_ms("Time")
                .alias("session_time_ms"),

            ns_to_ms("LapTime")
                .alias("lap_time_ms"),

            ns_to_ms("PitOutTime")
                .alias("pit_out_time_ms"),

            ns_to_ms("PitInTime")
                .alias("pit_in_time_ms"),

            ns_to_ms("Sector1Time")
                .alias("sector_1_time_ms"),

            ns_to_ms("Sector2Time")
                .alias("sector_2_time_ms"),

            ns_to_ms("Sector3Time")
                .alias("sector_3_time_ms"),

            ns_to_ms("Sector1SessionTime")
                .alias(
                    "sector_1_session_time_ms"
                ),

            ns_to_ms("Sector2SessionTime")
                .alias(
                    "sector_2_session_time_ms"
                ),

            ns_to_ms("Sector3SessionTime")
                .alias(
                    "sector_3_session_time_ms"
                ),

            # ----------------------------------------------
            # Speed
            # ----------------------------------------------

            clean_double("SpeedI1")
                .alias("speed_i1_kph"),

            clean_double("SpeedI2")
                .alias("speed_i2_kph"),

            clean_double("SpeedFL")
                .alias("speed_fl_kph"),

            clean_double("SpeedST")
                .alias("speed_st_kph"),

            # ----------------------------------------------
            # Tyres
            # ----------------------------------------------

            col("IsPersonalBest")
                .cast("boolean")
                .alias("is_personal_best"),

            upper_spark(
                clean_string("Compound")
            ).alias("compound"),

            double_to_int("TyreLife")
                .alias("tyre_life_laps"),

            col("FreshTyre")
                .cast("boolean")
                .alias("fresh_tyre"),

            # ----------------------------------------------
            # Lap start
            # ----------------------------------------------

            ns_to_ms("LapStartTime")
                .alias("lap_start_time_ms"),

            col("LapStartDate")
                .cast("timestamp_ntz")
                .alias("lap_start_date"),

            # ----------------------------------------------
            # Race state
            # ----------------------------------------------

            clean_string("TrackStatus")
                .alias("track_status"),

            double_to_int("Position")
                .alias("position"),

            # ----------------------------------------------
            # Quality flags
            # ----------------------------------------------

            when(
                col("Deleted").isNull(),
                lit(None).cast("boolean")
            )
            .otherwise(
                col("Deleted") == 1
            )
            .alias("is_deleted"),

            clean_string("DeletedReason")
                .alias("deleted_reason"),

            col("FastF1Generated")
                .cast("boolean")
                .alias("is_fastf1_generated"),

            col("IsAccurate")
                .cast("boolean")
                .alias("is_accurate"),

            # ----------------------------------------------
            # Lineage
            # ----------------------------------------------

            col("_source_file")
                .alias("source_file"),

            col("_source_file_modification_time")
                .alias("source_modified_at"),

            col("_ingested_at")
                .alias("bronze_ingested_at")
        )
    )

In [0]:
def validate_laps(df):
    validation = (
        df
        .agg(
            sum_spark(
                when(
                    col("season").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_season"),

            sum_spark(
                when(
                    col("round").isNull()
                    |
                    (col("round") <= 0),
                    1
                ).otherwise(0)
            ).alias("invalid_round"),

            sum_spark(
                when(
                    col("driver_id").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_driver_id"),

            sum_spark(
                when(
                    col("constructor_id").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_constructor_id"),

            sum_spark(
                when(
                    col("lap_number").isNull()
                    |
                    (col("lap_number") <= 0),
                    1
                ).otherwise(0)
            ).alias("invalid_lap_number"),

            sum_spark(
                when(
                    col("stint_number") < 1,
                    1
                ).otherwise(0)
            ).alias("invalid_stint"),

            sum_spark(
                when(
                    col("tyre_life_laps") < 0,
                    1
                ).otherwise(0)
            ).alias("invalid_tyre_life"),

            sum_spark(
                when(
                    col("position") < 1,
                    1
                ).otherwise(0)
            ).alias("invalid_position"),

            sum_spark(
                when(
                    col("lap_time_ms") <= 0,
                    1
                ).otherwise(0)
            ).alias("invalid_lap_time"),

            sum_spark(
                when(
                    col("speed_i1_kph") < 0,
                    1
                ).otherwise(0)
            ).alias("negative_speed_i1"),

            sum_spark(
                when(
                    col("speed_i2_kph") < 0,
                    1
                ).otherwise(0)
            ).alias("negative_speed_i2"),

            sum_spark(
                when(
                    col("speed_fl_kph") < 0,
                    1
                ).otherwise(0)
            ).alias("negative_speed_fl"),

            sum_spark(
                when(
                    col("speed_st_kph") < 0,
                    1
                ).otherwise(0)
            ).alias("negative_speed_st")
        )
        .first()
        .asDict()
    )

    errors = {
        rule: value or 0
        for rule, value in validation.items()
        if (value or 0) > 0
    }

    duplicate_laps = (
        df
        .groupBy(
            "season",
            "round",
            "driver_id",
            "lap_number"
        )
        .count()
        .filter(
            col("count") > 1
        )
        .count()
    )

    if duplicate_laps > 0:
        errors["duplicate_laps"] = (
            duplicate_laps
        )

    if errors:
        raise ValueError(
            f"Silver laps validation failed: {errors}"
        )

    print(
        f"Validation OK: {df.count()} laps ready for Silver."
    )

In [0]:
def add_laps_hash(df):
    business_columns = [
        "season",
        "round",
        "driver_id",
        "driver_number",
        "constructor_id",
        "lap_number",
        "stint_number",
        "session_time_ms",
        "lap_time_ms",
        "pit_out_time_ms",
        "pit_in_time_ms",
        "sector_1_time_ms",
        "sector_2_time_ms",
        "sector_3_time_ms",
        "sector_1_session_time_ms",
        "sector_2_session_time_ms",
        "sector_3_session_time_ms",
        "speed_i1_kph",
        "speed_i2_kph",
        "speed_fl_kph",
        "speed_st_kph",
        "is_personal_best",
        "compound",
        "tyre_life_laps",
        "fresh_tyre",
        "lap_start_time_ms",
        "lap_start_date",
        "track_status",
        "position",
        "is_deleted",
        "deleted_reason",
        "is_fastf1_generated",
        "is_accurate"
    ]

    hash_expression = concat_ws(
        "||",
        *[
            coalesce(
                col(column).cast("string"),
                lit("<NULL>")
            )
            for column in business_columns
        ]
    )

    return (
        df
        .withColumn(
            "record_hash",
            sha2(
                hash_expression,
                256
            )
        )
        .withColumn(
            "silver_updated_at",
            current_timestamp()
        )
    )

In [0]:
def build_race_scope_condition(df):
    races = (
        df
        .select(
            "season",
            "round"
        )
        .distinct()
        .collect()
    )

    if not races:
        raise ValueError(
            "No race data available for merge."
        )

    return " OR ".join(
        [
            (
                f"(target.season = {row['season']} "
                f"AND target.round = {row['round']})"
            )
            for row in races
        ]
    )

In [0]:
def merge_laps(df):
    if not spark.catalog.tableExists(
        SILVER_LAPS_TABLE
    ):
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(
                SILVER_LAPS_TABLE
            )
        )

        print(
            f"Created {SILVER_LAPS_TABLE}"
        )
        return

    target = DeltaTable.forName(
        spark,
        SILVER_LAPS_TABLE
    )

    race_scope = build_race_scope_condition(
        df
    )

    (
        target.alias("target")
        .merge(
            df.alias("source"),
            """
            target.season = source.season
            AND target.round = source.round
            AND target.driver_id = source.driver_id
            AND target.lap_number = source.lap_number
            """
        )
        .whenMatchedUpdateAll(
            condition="""
                target.record_hash <> source.record_hash
            """
        )
        .whenNotMatchedInsertAll()
        .whenNotMatchedBySourceDelete(
            condition=race_scope
        )
        .execute()
    )

    print(
        f"Merged data into {SILVER_LAPS_TABLE}"
    )

In [0]:
laps_source_df = (
    get_latest_laps_snapshot()
)

enriched_laps_df = (
    enrich_laps_with_participants(
        laps_source_df
    )
)

laps_df = transform_laps(
    enriched_laps_df
)

validate_laps(
    laps_df
)

laps_df = add_laps_hash(
    laps_df
)

merge_laps(
    laps_df
)

In [0]:
display(
    spark.table(
        SILVER_LAPS_TABLE
    )
    .filter(
        (col("season") == 2025)
        &
        (col("round") == 2)
        &
        (col("driver_id") == "piastri")
    )
    .select(
        "lap_number",
        "stint_number",
        "lap_time_ms",
        "sector_1_time_ms",
        "sector_2_time_ms",
        "sector_3_time_ms",
        "compound",
        "tyre_life_laps",
        "position",
        "is_personal_best",
        "is_deleted",
        "is_accurate"
    )
    .orderBy(
        "lap_number"
    )
)